# 🌾 Seasonal Agriculture Performance Analysis
### Major Project — VOIS AICTE Batch 1 (2026–2027)

**Project focus:** Analyze how agricultural performance varies across seasons and identify meaningful patterns, trends, relationships, differences, and unusual observations.

This notebook follows the project brief: dataset understanding, cleaning, seasonal comparison, environmental/resource/economic analysis, statistical techniques, visualization, evidence-based conclusions, and recommendations.


## 1. Project Problem Statement

Agricultural activities are influenced by seasonal variations in environmental conditions, farming practices, resource availability, and market conditions. This analysis investigates seasonal differences in agricultural performance using the supplied dataset.

### Main analytical questions
1. How does agricultural performance vary across seasons?
2. What major seasonal patterns can be observed?
3. Which characteristics change between seasons?
4. What differences exist in agricultural activities across seasons?
5. How does resource usage vary across seasons?
6. Are environmental conditions related to agricultural performance?
7. How do economic outcomes vary across seasons?
8. Are seasonal patterns consistent across regions or crops?
9. Are there unusual or unexpected seasonal observations?
10. What evidence-based recommendations can be made?


## 2. Import Libraries

The notebook uses **Pandas/NumPy** for data handling, **Matplotlib/Seaborn** for visualization, and **SciPy/Statsmodels/Scikit-learn** for statistical analysis and modeling.


In [ ]:
# ==============================
# BLOCK 1 — IMPORT LIBRARIES
# ==============================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


## 3. Load the Dataset

### Google Colab
The first option lets you upload the supplied CSV directly from your computer.

If you already placed the CSV in `/content/`, you can use the second option instead.


In [ ]:
# ==============================
# BLOCK 2 — LOAD DATASET IN COLAB
# ==============================

from google.colab import files
uploaded = files.upload()

# Automatically find the uploaded CSV file
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("No CSV file was uploaded.")

file_path = csv_files[0]
df = pd.read_csv(file_path)

print(f"Loaded file: {file_path}")
print(f"Dataset shape: {df.shape}")


In [ ]:
# Alternative if the CSV is already in /content/
# df = pd.read_csv("/content/seasonal_agriculture_performance_dataset.csv")
# print(df.shape)


## 4. Initial Dataset Understanding

The supplied dataset contains **4,000 records and 28 columns**. It includes farm/location information, crop and season, environmental conditions, inputs/resources, yield, water efficiency, market price, cost, revenue, profit, and disease/pest risk.


In [ ]:
# ==============================
# BLOCK 3 — INITIAL INSPECTION
# ==============================

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("Data_Type"))


In [ ]:
# ==============================
# BLOCK 4 — SUMMARY INFORMATION
# ==============================

print("Dataset information:")
df.info()

print("\nNumerical summary:")
display(df.describe().T)

print("\nCategorical summary:")
display(df.describe(include="object").T)


## 5. Data Quality Check

The project brief requires data cleaning and preparation. We therefore check:
- missing values
- duplicate rows
- incorrect data types
- invalid numeric values
- suspicious values/outliers


In [ ]:
# ==============================
# BLOCK 5 — MISSING VALUES
# ==============================

missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

missing = missing[missing["Missing_Count"] > 0].sort_values(
    "Missing_Count", ascending=False
)

if missing.empty:
    print("No missing values found.")
else:
    display(missing)


In [ ]:
# ==============================
# BLOCK 6 — DUPLICATES
# ==============================

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate rows to remove.")


In [ ]:
# ==============================
# BLOCK 7 — DATA TYPE & UNIQUE-VALUE CHECK
# ==============================

categorical_cols = df.select_dtypes(include="object").columns.tolist()
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

print("Categorical columns:", categorical_cols)
print("\nNumeric columns:", numeric_cols)

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(20))


## 6. Cleaning and Preparation

For numeric variables, missing observations are imputed using the **median**, which is less sensitive to extreme values than the mean. For categorical variables, missing observations are filled with the **mode**.

The original dataset is preserved in `df_raw` so the cleaning process remains transparent.


In [ ]:
# ==============================
# BLOCK 8 — CLEAN DATA
# ==============================

df_raw = df.copy()

# Numeric columns: median imputation
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Categorical columns: mode imputation
for col in categorical_cols:
    if df[col].isna().any():
        mode_value = df[col].mode(dropna=True)
        if len(mode_value) > 0:
            df[col] = df[col].fillna(mode_value.iloc[0])

print("Remaining missing values:", int(df.isna().sum().sum()))


In [ ]:
# ==============================
# BLOCK 9 — LOGICAL VALIDATION
# ==============================

# Display potentially invalid negative values in variables that should not be negative.
non_negative_cols = [
    "Farm_Area_Hectares", "Rainfall_mm", "Humidity_pct",
    "Sunlight_Hours_Day", "Soil_Moisture_pct",
    "Nitrogen_kg_ha", "Phosphorus_kg_ha", "Potassium_kg_ha",
    "Fertilizer_kg_ha", "Pesticide_Litre_ha",
    "Seed_Quality_Score", "Yield_Tonnes_Ha",
    "Production_Tonnes", "Market_Price_INR_Tonne",
    "Total_Cost_INR", "Revenue_INR", "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3", "Disease_Pest_Risk_pct"
]

for col in non_negative_cols:
    if col in df.columns:
        bad = (df[col] < 0).sum()
        if bad:
            print(f"{col}: {bad} negative values found")

print("\nUnique seasons:", df["Season"].unique())
print("Unique crops:", df["Crop"].unique())
print("Unique irrigation methods:", df["Irrigation_Method"].unique())


## 7. Feature Engineering

The project focuses on agricultural performance. We create useful derived measures while retaining the original variables.

- **Profit_Margin_pct** = Profit / Revenue × 100
- **Cost_per_Tonne** = Total Cost / Production
- **Revenue_per_Hectare** = Revenue / Farm Area
- **Profit_per_Hectare** = Profit / Farm Area
- **Water_Productivity** = Production / Water Used × 1000

These are analytical indicators, not replacements for the original dataset columns.


In [ ]:
# ==============================
# BLOCK 10 — FEATURE ENGINEERING
# ==============================

eps = 1e-9

df["Profit_Margin_pct"] = np.where(
    df["Revenue_INR"].abs() > eps,
    (df["Profit_INR"] / df["Revenue_INR"]) * 100,
    np.nan
)

df["Cost_per_Tonne"] = np.where(
    df["Production_Tonnes"] > eps,
    df["Total_Cost_INR"] / df["Production_Tonnes"],
    np.nan
)

df["Revenue_per_Hectare"] = np.where(
    df["Farm_Area_Hectares"] > eps,
    df["Revenue_INR"] / df["Farm_Area_Hectares"],
    np.nan
)

df["Profit_per_Hectare"] = np.where(
    df["Farm_Area_Hectares"] > eps,
    df["Profit_INR"] / df["Farm_Area_Hectares"],
    np.nan
)

df["Water_Productivity_t_per_1000m3"] = np.where(
    df["Water_Used_m3"] > eps,
    (df["Production_Tonnes"] / df["Water_Used_m3"]) * 1000,
    np.nan
)

display(df[[
    "Season", "Yield_Tonnes_Ha", "Production_Tonnes",
    "Profit_INR", "Profit_Margin_pct",
    "Revenue_per_Hectare", "Profit_per_Hectare",
    "Water_Efficiency_t_per_1000m3",
    "Water_Productivity_t_per_1000m3"
]].head())


## 8. Seasonal Distribution

Before comparing performance, inspect how many observations belong to each season. This prevents interpreting a season without considering its sample size.


In [ ]:
# ==============================
# BLOCK 11 — SEASON COUNTS
# ==============================

season_counts = df["Season"].value_counts().sort_index()
display(season_counts.to_frame("Farm_Records"))

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Season", order=season_counts.index)
plt.title("Number of Farm Records by Season")
plt.xlabel("Season")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()


## 9. Seasonal Performance Comparison

The main performance indicators are:
- yield per hectare
- production
- revenue
- profit
- profit margin
- water efficiency
- disease/pest risk


In [ ]:
# ==============================
# BLOCK 12 — SEASONAL PERFORMANCE TABLE
# ==============================

performance_cols = [
    "Yield_Tonnes_Ha",
    "Production_Tonnes",
    "Revenue_INR",
    "Profit_INR",
    "Profit_Margin_pct",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

season_summary = df.groupby("Season")[performance_cols].agg(
    ["count", "mean", "median", "std"]
).round(2)

display(season_summary)


In [ ]:
# ==============================
# BLOCK 13 — SEASONAL PERFORMANCE BOXPLOTS
# ==============================

plot_vars = [
    "Yield_Tonnes_Ha",
    "Profit_INR",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

for col in plot_vars:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x="Season", y=col)
    plt.title(f"{col} by Season")
    plt.xlabel("Season")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 10. Crop × Season Analysis

Seasonal performance may differ because different crops have different production and economic characteristics. We therefore compare crops within seasons rather than relying only on overall averages.


In [ ]:
# ==============================
# BLOCK 14 — CROP-SEASON PERFORMANCE
# ==============================

crop_season = df.groupby(["Season", "Crop"]).agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Risk=("Disease_Pest_Risk_pct", "mean")
).reset_index()

display(crop_season.sort_values(["Season", "Avg_Yield"], ascending=[True, False]))


In [ ]:
# ==============================
# BLOCK 15 — HEATMAP: YIELD BY CROP AND SEASON
# ==============================

yield_pivot = df.pivot_table(
    index="Crop",
    columns="Season",
    values="Yield_Tonnes_Ha",
    aggfunc="mean"
)

plt.figure(figsize=(9, 6))
sns.heatmap(yield_pivot, annot=True, fmt=".2f")
plt.title("Average Yield by Crop and Season")
plt.xlabel("Season")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()


## 11. Regional / State Analysis

The project brief asks whether seasonal patterns are consistent across regions. We compare state-level seasonal performance and identify the strongest/weakest combinations for further investigation.


In [ ]:
# ==============================
# BLOCK 16 — STATE × SEASON ANALYSIS
# ==============================

state_season = df.groupby(["State", "Season"]).agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Risk=("Disease_Pest_Risk_pct", "mean")
).reset_index()

display(state_season.sort_values(["Season", "Avg_Profit"], ascending=[True, False]).head(30))


In [ ]:
# ==============================
# BLOCK 17 — STATE × SEASON YIELD HEATMAP
# ==============================

state_yield_pivot = df.pivot_table(
    index="State",
    columns="Season",
    values="Yield_Tonnes_Ha",
    aggfunc="mean"
)

plt.figure(figsize=(10, 7))
sns.heatmap(state_yield_pivot, annot=True, fmt=".2f")
plt.title("Average Yield by State and Season")
plt.xlabel("Season")
plt.ylabel("State")
plt.tight_layout()
plt.show()


## 12. Environmental Conditions Across Seasons

The project specifically asks whether environmental conditions differ by season and whether they relate to agricultural outcomes.


In [ ]:
# ==============================
# BLOCK 18 — ENVIRONMENTAL SEASONAL COMPARISON
# ==============================

environment_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct"
]

environment_summary = df.groupby("Season")[environment_cols].mean().round(2)
display(environment_summary)


In [ ]:
# ==============================
# BLOCK 19 — ENVIRONMENTAL BOXPLOTS
# ==============================

for col in environment_cols:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x="Season", y=col)
    plt.title(f"{col} by Season")
    plt.xlabel("Season")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 13. Resource Usage by Season

Compare irrigation, fertilizer, pesticide, nutrients, and water use to understand how resource requirements vary across seasons.


In [ ]:
# ==============================
# BLOCK 20 — RESOURCE USAGE
# ==============================

resource_cols = [
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Water_Used_m3"
]

resource_summary = df.groupby("Season")[resource_cols].agg(
    ["mean", "median"]
).round(2)

display(resource_summary)


In [ ]:
# ==============================
# BLOCK 21 — IRRIGATION METHOD BY SEASON
# ==============================

irrigation_table = pd.crosstab(
    df["Season"],
    df["Irrigation_Method"],
    normalize="index"
).mul(100).round(2)

display(irrigation_table)

plt.figure(figsize=(10, 5))
irrigation_table.plot(kind="bar", figsize=(10, 5))
plt.title("Irrigation Method Distribution within Each Season")
plt.xlabel("Season")
plt.ylabel("Percentage of Records")
plt.xticks(rotation=0)
plt.legend(title="Irrigation Method")
plt.tight_layout()
plt.show()


## 14. Economic Performance

Agricultural performance is not only yield. Economic outcomes are examined using revenue, cost, profit, profit margin, and per-hectare indicators.


In [ ]:
# ==============================
# BLOCK 22 — ECONOMIC ANALYSIS
# ==============================

economic_cols = [
    "Market_Price_INR_Tonne",
    "Total_Cost_INR",
    "Revenue_INR",
    "Profit_INR",
    "Profit_Margin_pct",
    "Revenue_per_Hectare",
    "Profit_per_Hectare"
]

economic_summary = df.groupby("Season")[economic_cols].agg(
    ["mean", "median"]
).round(2)

display(economic_summary)


In [ ]:
# ==============================
# BLOCK 23 — PROFIT BY SEASON
# ==============================

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Season", y="Profit_INR")
plt.axhline(0, linestyle="--")
plt.title("Profit Distribution by Season")
plt.xlabel("Season")
plt.ylabel("Profit (INR)")
plt.tight_layout()
plt.show()


## 15. Water Efficiency

Water efficiency is important for understanding the relationship between production and water use.

The dataset already provides `Water_Efficiency_t_per_1000m3`; the notebook also calculates an independent `Water_Productivity_t_per_1000m3` from production and water used.


In [ ]:
# ==============================
# BLOCK 24 — WATER EFFICIENCY ANALYSIS
# ==============================

water_summary = df.groupby("Season").agg(
    Avg_Water_Used_m3=("Water_Used_m3", "mean"),
    Avg_Provided_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Calculated_Water_Productivity=("Water_Productivity_t_per_1000m3", "mean")
).round(3)

display(water_summary)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Season", y="Water_Efficiency_t_per_1000m3")
plt.title("Water Efficiency by Season")
plt.xlabel("Season")
plt.ylabel("Water Efficiency (t per 1000 m³)")
plt.tight_layout()
plt.show()


## 16. Correlation Analysis

Correlation helps identify variables that move together. Correlation does **not** by itself prove causation.

Spearman correlation is included because agricultural variables can have skewed distributions and non-linear monotonic relationships.


In [ ]:
# ==============================
# BLOCK 25 — CORRELATION MATRIX
# ==============================

corr_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct",
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Seed_Quality_Score",
    "Yield_Tonnes_Ha",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct",
    "Market_Price_INR_Tonne",
    "Profit_INR"
]

corr = df[corr_cols].corr(method="spearman")

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Spearman Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# BLOCK 26 — TOP CORRELATIONS WITH YIELD
# ==============================

yield_corr = corr["Yield_Tonnes_Ha"].drop("Yield_Tonnes_Ha").sort_values(
    key=lambda s: s.abs(), ascending=False
)

display(yield_corr.to_frame("Spearman_Correlation_with_Yield"))


## 17. Statistical Test: Seasonal Yield Difference

Because there are three seasons, an omnibus test is appropriate for checking whether yield distributions differ.

We report:
- **Kruskal–Wallis H-test**: non-parametric comparison across the three seasons.
- Effect interpretation should consider practical importance, not only p-values.


In [ ]:
# ==============================
# BLOCK 27 — KRUSKAL-WALLIS TEST FOR YIELD
# ==============================

season_groups = [
    group["Yield_Tonnes_Ha"].dropna().values
    for _, group in df.groupby("Season")
]

season_names = list(df["Season"].dropna().unique())

h_stat, p_value = stats.kruskal(*season_groups)

print("Kruskal-Wallis H statistic:", round(h_stat, 4))
print("p-value:", round(p_value, 6))

if p_value < 0.05:
    print("Result: statistically significant evidence of a difference in yield among at least one pair of seasons (α = 0.05).")
else:
    print("Result: no statistically significant evidence of a yield difference among seasons at α = 0.05.")


## 18. Post-hoc Seasonal Comparison

If the omnibus Kruskal–Wallis test is significant, pairwise Mann–Whitney U tests are performed with Bonferroni correction.

If it is not significant, the pairwise results are not used to claim seasonal differences.


In [ ]:
# ==============================
# BLOCK 28 — POST-HOC PAIRWISE TESTS
# ==============================

from itertools import combinations

season_list = sorted(df["Season"].dropna().unique())
pairwise_results = []

for s1, s2 in combinations(season_list, 2):
    x = df.loc[df["Season"] == s1, "Yield_Tonnes_Ha"].dropna()
    y = df.loc[df["Season"] == s2, "Yield_Tonnes_Ha"].dropna()

    u, p = stats.mannwhitneyu(x, y, alternative="two-sided")

    pairwise_results.append({
        "Season_1": s1,
        "Season_2": s2,
        "U_statistic": u,
        "Raw_p_value": p,
        "Bonferroni_p_value": min(p * len(pairwise_results) if False else p, 1.0)
    })

pairwise_df = pd.DataFrame(pairwise_results)

# Correct Bonferroni adjustment for the number of comparisons
m = len(pairwise_df)
pairwise_df["Bonferroni_p_value"] = (pairwise_df["Raw_p_value"] * m).clip(upper=1)

display(pairwise_df)

if p_value < 0.05:
    print("Because the omnibus test is significant, inspect the adjusted pairwise p-values above.")
else:
    print("Omnibus test was not significant; do not claim pairwise seasonal differences as statistically established.")


## 19. Relationship Between Environment and Yield

We examine selected environmental variables against yield using scatter plots and Spearman correlation.


In [ ]:
# ==============================
# BLOCK 29 — ENVIRONMENT VS YIELD
# ==============================

environment_yield_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct"
]

for col in environment_yield_cols:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x=col, y="Yield_Tonnes_Ha", alpha=0.5)
    sns.regplot(
        data=df, x=col, y="Yield_Tonnes_Ha",
        scatter=False, ci=None, line_kws={"linewidth": 2}
    )
    plt.title(f"{col} vs Yield")
    plt.xlabel(col)
    plt.ylabel("Yield (Tonnes/Ha)")
    plt.tight_layout()
    plt.show()

    r, p = stats.spearmanr(df[col], df["Yield_Tonnes_Ha"], nan_policy="omit")
    print(f"{col}: Spearman r = {r:.3f}, p = {p:.6f}")


## 20. Disease/Pest Risk Analysis

The brief explicitly asks about unusual patterns and seasonal environmental/performance relationships. Disease/pest risk is therefore compared by season and crop.


In [ ]:
# ==============================
# BLOCK 30 — DISEASE/PEST RISK
# ==============================

risk_summary = df.groupby("Season")["Disease_Pest_Risk_pct"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).round(2)

display(risk_summary)

risk_crop = df.groupby(["Season", "Crop"])["Disease_Pest_Risk_pct"].mean().reset_index()

display(
    risk_crop.sort_values(
        "Disease_Pest_Risk_pct", ascending=False
    ).head(20)
)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Season", y="Disease_Pest_Risk_pct")
plt.title("Disease/Pest Risk by Season")
plt.xlabel("Season")
plt.ylabel("Disease/Pest Risk (%)")
plt.tight_layout()
plt.show()


## 21. Outlier / Unusual Observation Detection

The project asks for unusual or unexpected seasonal patterns. We use the IQR rule as a screening method.

Important: an outlier is **not automatically an error**. It may represent a genuine farm-level observation.


In [ ]:
# ==============================
# BLOCK 31 — IQR OUTLIER SCREENING
# ==============================

outlier_cols = [
    "Yield_Tonnes_Ha",
    "Production_Tonnes",
    "Profit_INR",
    "Water_Efficiency_t_per_1000m3"
]

outlier_summary = []

for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (df[col] < lower) | (df[col] > upper)

    outlier_summary.append({
        "Variable": col,
        "Q1": q1,
        "Q3": q3,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(mask.sum()),
        "Outlier_Percentage": round(mask.mean() * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_summary)
display(outlier_summary)


## 22. Inspect the Most Extreme Yield Records

This is an investigation step, not an automatic deletion step. The records are displayed with farm, location, crop, season, production, revenue, profit, and environmental context.


In [ ]:
# ==============================
# BLOCK 32 — EXTREME YIELD RECORDS
# ==============================

extreme_yield = df.nlargest(15, "Yield_Tonnes_Ha")[
    [
        "Farm_ID", "State", "District", "Crop", "Season",
        "Farm_Area_Hectares", "Rainfall_mm",
        "Avg_Temperature_C", "Soil_Moisture_pct",
        "Yield_Tonnes_Ha", "Production_Tonnes",
        "Revenue_INR", "Profit_INR",
        "Water_Used_m3", "Water_Efficiency_t_per_1000m3",
        "Disease_Pest_Risk_pct"
    ]
]

display(extreme_yield)


## 23. Optional Predictive Model: Yield Drivers

A Random Forest regression model is included as an **exploratory analytical extension**, not as proof of causation.

The model estimates yield from environmental, resource, crop, season, irrigation, and farm variables. One-hot encoding is used for categorical variables.


In [ ]:
# ==============================
# BLOCK 33 — PREPARE MODEL DATA
# ==============================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

model_features = [
    "State", "District", "Crop", "Season",
    "Farm_Area_Hectares", "Rainfall_mm",
    "Avg_Temperature_C", "Humidity_pct",
    "Sunlight_Hours_Day", "Soil_pH",
    "Soil_Moisture_pct", "Nitrogen_kg_ha",
    "Phosphorus_kg_ha", "Potassium_kg_ha",
    "Irrigation_Method", "Fertilizer_kg_ha",
    "Pesticide_Litre_ha", "Seed_Quality_Score",
    "Water_Used_m3", "Disease_Pest_Risk_pct"
]

X = df[model_features].copy()
y = df["Yield_Tonnes_Ha"].copy()

cat_features = X.select_dtypes(include="object").columns.tolist()
num_features = X.select_dtypes(exclude="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


In [ ]:
# ==============================
# BLOCK 34 — TRAIN & EVALUATE MODEL
# ==============================

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest Yield Model")
print("-------------------------")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


In [ ]:
# ==============================
# BLOCK 35 — PREDICTED VS ACTUAL
# ==============================

plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.title("Actual vs Predicted Yield")
plt.xlabel("Actual Yield (Tonnes/Ha)")
plt.ylabel("Predicted Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# BLOCK 36 — FEATURE IMPORTANCE
# ==============================

rf = model.named_steps["regressor"]
prep = model.named_steps["preprocessor"]

feature_names = prep.get_feature_names_out()
importances = rf.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

display(feature_importance.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 24. Automated Evidence Summary

This block creates a compact table of key seasonal indicators. Use these values when writing the final report rather than manually copying numbers from charts.


In [ ]:
# ==============================
# BLOCK 37 — FINAL EVIDENCE TABLE
# ==============================

final_evidence = df.groupby("Season").agg(
    Records=("Farm_ID", "count"),
    Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
    Median_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "median"),
    Avg_Production_Tonnes=("Production_Tonnes", "mean"),
    Avg_Revenue_INR=("Revenue_INR", "mean"),
    Avg_Profit_INR=("Profit_INR", "mean"),
    Avg_Profit_Margin_pct=("Profit_Margin_pct", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct", "mean")
).round(2)

display(final_evidence)


## 25. Automatically Identify Best/Worst Seasonal Indicators

These outputs are descriptive only. They should be interpreted alongside sample size, variability, crop mix, and statistical testing.


In [ ]:
# ==============================
# BLOCK 38 — BEST/WORST SEASONAL INDICATORS
# ==============================

indicators = {
    "Average Yield": ("Yield_Tonnes_Ha", "max"),
    "Average Profit": ("Profit_INR", "max"),
    "Average Water Efficiency": ("Water_Efficiency_t_per_1000m3", "max"),
    "Average Disease/Pest Risk": ("Disease_Pest_Risk_pct", "min")
}

for label, (col, direction) in indicators.items():
    values = df.groupby("Season")[col].mean()
    selected = values.idxmax() if direction == "max" else values.idxmin()
    print(f"{label}: {selected} ({values[selected]:,.2f})")


## 26. Final Conclusions — Evidence-Based Template

Run the previous blocks first. Then replace the placeholders below using the actual results produced by this notebook.

### Conclusion checklist
- State which season has the highest/lowest average yield.
- State whether the statistical test supports a significant seasonal yield difference.
- Identify important environmental differences.
- Identify important resource-use differences.
- Compare economic performance.
- Mention crop/state differences where supported.
- Mention unusual observations without automatically treating them as errors.
- Distinguish correlation/prediction from causation.

### Recommendations should be tied directly to evidence
Recommendations may concern:
- seasonal planning
- resource allocation
- irrigation strategy
- crop/region prioritization
- risk monitoring
- further data collection


## 27. Project Deliverables Checklist

This notebook covers the requirements stated in the project brief:

✅ Dataset exploration  
✅ Data cleaning and preparation  
✅ Seasonal performance comparison  
✅ Seasonal patterns and trends  
✅ Environmental condition analysis  
✅ Resource usage analysis  
✅ Economic outcome analysis  
✅ Crop and regional comparisons  
✅ Statistical testing  
✅ Visualization  
✅ Correlation analysis  
✅ Unusual/outlier observation screening  
✅ Evidence-based interpretation  
✅ Data-driven recommendation framework  
✅ Jupyter/Google Colab notebook documentation


# 🌱 End of Analysis

**Important:** Do not write conclusions before running the analysis. The final report should use the numerical results generated from the supplied dataset and should avoid unsupported causal claims.
